### Установка библиотек

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix

from datasets import load_dataset
import random
import numpy as np
import json

from transformers import BertTokenizerFast, BertForSequenceClassification, TrainingArguments, Trainer, get_cosine_schedule_with_warmup

In [ ]:
def set_random_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_random_seed(224)

### Загрузка датасета для анализа тональности

In [ ]:
ds = load_dataset("ai-forever/kinopoisk-sentiment-classification")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### Токенизатор (и модель DeepPavlov)

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('DeepPavlov/rubert-base-cased')

### Реализация срезов

In [ ]:
def slice_token(index, sentences, labels, tokenizer, max_length):
    start, stop, step = index.indices(len(sentences))
    result = []
    for i in range(start, stop, step):
        encoding = tokenizer(
                sentences[i],
                padding='max_length',
                truncation = True,
                max_length = max_length,
                return_tensors = 'pt'
            )
        encoding['labels'] = [labels[i]]
        result.append({key : value[0] for key, value in encoding.items()})

    return result

### Кастомный датасет

In [ ]:
class SemDataset(Dataset):
    def __init__(self, sentences, labels, tokenizer, max_length):
        self.sentences = sentences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):

        if isinstance(idx, slice):
            return slice_token(idx, self.sentences, self.labels, self.tokenizer, self.max_length)
        elif isinstance(idx, int):
            tokens = self.sentences[idx]
            tag = self.labels[idx]

            encoding = self.tokenizer(
                tokens,
                padding='max_length',
                truncation = True,
                max_length = self.max_length,
                return_tensors = 'pt'
            )
            encoding['labels'] = [tag]
            return {key : value[0] for key, value in encoding.items()}

### Параметры

In [ ]:
batch_size = 16
max_length = 512
epochs = 3

### Создание датасетов для тренировки, валидации и тестирования

In [ ]:
dataset_train = SemDataset(ds['train']['text'], ds['train']['label'], tokenizer, max_length)
dataset_test = SemDataset(ds['test']['text'], ds['test']['label'], tokenizer, max_length)
dataset_val = SemDataset(ds['validation']['text'], ds['validation']['label'], tokenizer, max_length)

### Даталоадеры

In [ ]:
test_loader = DataLoader(dataset_test, batch_size, pin_memory=True)
train_loader = DataLoader(dataset_train, batch_size)
val_loader = DataLoader(dataset_val, batch_size)

### Предобученная модель DeepPavlov/rubert-base-cased

In [ ]:
num_labels = len(set(ds['train']['label']))
model = BertForSequenceClassification.from_pretrained('DeepPavlov/rubert-base-cased', num_labels= num_labels)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

### Функция для подсчета метрик

In [ ]:
def compute_metrics(p):

    predictions, labels = p

    predictions = predictions.argmax(axis=-1)
    cm = confusion_matrix(labels, predictions)

    print("Confusion Matrix:\n", cm)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

### Обучение с оптимизатором и шедулером

In [ ]:
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()))
total_steps = len(dataset_train) * epochs * batch_size

scheduler = get_cosine_schedule_with_warmup(
  optimizer,
  num_warmup_steps=total_steps*0.05,
  num_training_steps=total_steps
)

learning_rate = 2e-5
weight_decay = 0.01

### Задаем директории для сохранения чекпоинтов, конфигураций и метрик

In [ ]:
import os
outputs_dir = os.mkdir("../outputs", exist_ok=True)
model_dir = os.mkdir("../model_tokenizer", exist_ok=True)
checkpoints_dir = os.path.join(model_dir, 'checkpoints')

### Агрументы

In [ ]:
training_args = TrainingArguments(
    output_dir=checkpoints_dir,
    eval_strategy="epoch",
    learning_rate = learning_rate,
    weight_decay = weight_decay,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    logging_steps=10,
    save_strategy="epoch", # возможно сократить
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    gradient_checkpointing=True # что и зачем
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_loader.dataset,
    eval_dataset=val_loader.dataset,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, scheduler)
    )

In [ ]:
train_metrics = trainer.train().metrics

with open(os.path.join(outputs_dir, "train_metrics.json"), "w") as f:
    json.dump(train_metrics, f, indent=2)

eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

with open(os.path.join(outputs_dir, "eval_metrics.json"), "w") as f:
    json.dump(eval_results, f, indent=2)

trainer.save_model("../model") # поменять
tokenizer.save_pretrained("../model_tokenizer")

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.737192,1.169393,0.467333,0.414193,0.554455,0.467333
2,0.747161,0.778621,0.654667,0.641924,0.654190,0.654667
3,0.608344,0.757233,0.632000,0.640688,0.665971,0.632000


Confusion Matrix:
 [[119 245 136]
 [ 11  95 394]
 [  0  13 487]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Confusion Matrix:
 [[465  33   2]
 [196 216  88]
 [ 40 159 301]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Confusion Matrix:
 [[465  33   2]
 [192 219  89]
 [ 38 161 301]]
Evaluation Results: {'eval_loss': 0.7756486535072327, 'eval_accuracy': 0.6566666666666666, 'eval_f1': 0.6442892335134965, 'eval_precision': 0.6557294116280824, 'eval_recall': 0.6566666666666667, 'eval_runtime': 46.1845, 'eval_samples_per_second': 32.478, 'eval_steps_per_second': 2.035, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('tokenizer/final_tokenizer/tokenizer_config.json',
 'tokenizer/final_tokenizer/tokenizer.json')

In [ ]:
test_results = trainer.evaluate(eval_dataset=test_loader.dataset, metric_key_prefix="test")
print(f"Evaluation Results: {test_results}")

with open("../outputs/test_metrics.json", "w") as f:
    json.dump(test_results, f, indent=2)

### Тестирование модели

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
def test_model(model, data_loader, device):
  model = model.eval()

  all_preds = torch.tensor([], device=device)
  all_trues = torch.tensor([], device=device)

  with torch.no_grad():
    for d in data_loader:
      input_ids = d["input_ids"].to(device)
      attention_mask = d["attention_mask"].to(device)
      targets = d["labels"].to(device)

      outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
      )
      preds = torch.argmax(outputs['logits'], axis=-1)
      all_preds = torch.cat((all_preds, preds), -1)
      all_trues = torch.cat((all_trues, targets), -1)

  precision, recall, f1, _ = precision_recall_fscore_support(all_trues.cpu(), all_preds.cpu(), average='macro')
  acc = accuracy_score(all_trues.cpu(), all_preds.cpu())
  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall
  }

### Результаты на тестовом датасете

In [ ]:
test = test_model(model, test_loader, device)
test